# 💼 The Analyst's Notebook · Part 5
### The first model, and whether you are allowed to believe it

Part 4 finished with a question written down precisely enough to answer: a target, three features that cannot see the future, a split by date, two baselines and a metric. The one thing it never did was fit anything.

That is what Part 5 starts with, and it takes about four lines. The rest of this notebook is the harder half: deciding which of several models to keep, without letting the test years anywhere near the decision.

It ends with the test Part 4's simple rule failed. A rule that looked convincing on Apple was positive on only 4 of the eleven instruments. The model chosen here has to do better than that across the whole desk, or it has not earned anything.

## How to work through this

- Run the **quick load** cell first. It brings back what Part 4 established and loads the price table.
- Each question builds on the last, so keep them in order and keep your variables. Later questions use the names earlier ones created.
- Cells with `...` are blanks. The notebook runs cleanly even before you fill them in, so **Run all** is always safe.
- Hints and solutions are folded under each question. Work first, then check.

**A note on units.** The lecture worked in percent so the numbers read on a slide. This notebook keeps the plain decimals of Parts 1 to 4, so an RMSE of `0.004` means 0.4 percentage points of daily volatility. Everything else is identical.

*Stuck for more than 15 minutes? Ask a friend, ask an AI for a hint (not the answer), or email me at `jobo@econ.au.dk`.*

---

## ⚙️ Quick load

The packages, the price table, and what Part 4 left you. Run it and read what it prints.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit

CANDIDATE_DIRS = ["data", os.path.join("..", "data"), "."]
REPO_RAW_URL = "https://raw.githubusercontent.com/theill95/mlfin-2026/main/data/"   # used when the CSV files are not next to the notebook


def data_path(filename):
    """Where the course CSV files are, wherever you happen to be running."""
    for folder in CANDIDATE_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    if REPO_RAW_URL is not None:
        return REPO_RAW_URL + filename
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from the course folder, "
        f"upload the CSV into Colab, or set REPO_RAW_URL."
    )


# The whole universe: eleven instruments, 2015 to 2024
prices = pd.read_csv(data_path("prices.csv"), parse_dates=["date"])
wide = prices.pivot(index="date", columns="ticker", values="close")
rets = wide.pct_change()
TICKERS = sorted(prices["ticker"].unique())

# --- What Part 4 established ---
part4_target = "sd of daily returns over the next 20 trading days"
part4_features = ["vol_20d", "ret_20d", "up_20d"]
part4_split = "by date: train to 2022-12-31, test from 2023-01-01"
part4_base_rmse = 0.00514      # predict the training average
part4_pers_rmse = 0.00417      # repeat the last 20 days
part4_pers_r2 = 0.340         # the bar a model has to clear

print("Loaded prices:", prices.shape[0], "rows")
print("Instruments  :", ", ".join(TICKERS))
print()
print("Part 4 left you a fully specified problem:")
print("  target  :", part4_target)
print("  features:", part4_features)
print("  split   :", part4_split)
print()
print("and two rules that use no model at all:")
print(f"  predict the average     RMSE {part4_base_rmse:.5f}")
print(f"  repeat the last 20 days RMSE {part4_pers_rmse:.5f}   R2 {part4_pers_r2:.3f}")
print()
print("Anything you build today has to beat both of those.")

---

### Q1 · Rebuild the table

Start from where Part 4 finished. Build the learning table for Apple again: the three features, then the target, with incomplete rows dropped. Call it `table`.

The features look back over the last twenty trading days. The target looks forward over the next twenty.

$$\text{vol\_next}_t = \text{sd}\big(r_{t+1},\, \ldots,\, r_{t+20}\big)$$

In [ ]:
table = ...
table

<details>
<summary>💡 Hint 1</summary>

The three features are `rets['AAPL'].rolling(20).std()`, `.rolling(20).mean()`, and `(rets['AAPL'] > 0).rolling(20).mean()`.

</details>

<details>
<summary>💡 Hint 2</summary>

The target is the first of those shifted backwards by twenty rows: `.shift(-20)`. Finish with `.dropna()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
table = pd.DataFrame({
    'vol_20d': rets['AAPL'].rolling(20).std(),
    'ret_20d': rets['AAPL'].rolling(20).mean(),
    'up_20d': (rets['AAPL'] > 0).rolling(20).mean(),
})
table['vol_next'] = rets['AAPL'].rolling(20).std().shift(-20)
table = table.dropna()
table
```

2,476 rows and four columns, exactly as in Part 4. Twenty rows are lost at the start, where the window had not filled, and twenty at the end, where the target reaches past the last date in the file.

</details>

---

### Q2 · The split, unchanged

Cut the table at the same date Part 4 chose: everything up to the end of 2022 for training, 2023 onwards for the test block. Call them `train` and `test`.

In [ ]:
train = ...
test = ...

print('train:', ...)
print('test :', ...)

<details>
<summary>💡 Hint</summary>

`table.loc[:'2022-12-31']` and `table.loc['2023-01-01':]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
train = table.loc[:'2022-12-31']
test = table.loc['2023-01-01':]

print('train:', len(train))
print('test :', len(test))
```

1,994 training rows and 482 test rows.

From here to Q13, the word `test` does not appear again. That is not an accident: every decision in between is made without it.

</details>

---

### Q3 · Fit the first model of the course

Fit a linear regression that predicts `vol_next` from `vol_20d` alone, on the training rows. Then print what it learned.

In [ ]:
X_train = ...
y_train = ...

model = LinearRegression()

...        # fit it

print('intercept:', ...)
print('coefficient:', ...)

<details>
<summary>💡 Hint 1</summary>

`X` needs a list of column names inside the brackets so it stays a table: `train[['vol_20d']]`. `y` is one column: `train['vol_next']`.

</details>

<details>
<summary>💡 Hint 2</summary>

`model.fit(X_train, y_train)`, then `model.intercept_` and `model.coef_`.

</details>

<details>
<summary>✅ Solution</summary>

```python
X_train = train[['vol_20d']]
y_train = train['vol_next']

model = LinearRegression()
model.fit(X_train, y_train)

print('intercept:', model.intercept_)
print('coefficient:', model.coef_)
```

An intercept of about 0.00876 and a slope of about 0.486.

Read the slope out loud. It is not 1. A month that is unusually volatile is followed by a month only about half as far from normal, which is mean reversion. Four sessions of setup, and the first model you fit tells you something true about markets.

</details>

---

### Q4 · Score it against Part 4's two rules

Predict on the test block and compute the RMSE. Put it next to the two numbers the quick load restored.

$$\text{RMSE}=\sqrt{\tfrac{1}{n}\sum_i (y_i-\hat{y}_i)^2}$$

In [ ]:
predictions = ...
model_rmse = ...

print('model      :', ...)
print('persistence:', part4_pers_rmse)
print('average    :', part4_base_rmse)

<details>
<summary>💡 Hint 1</summary>

`model.predict(test[['vol_20d']])`, with the same double brackets as the fit.

</details>

<details>
<summary>💡 Hint 2</summary>

`np.sqrt(mean_squared_error(test['vol_next'], predictions))`, true values first.

</details>

<details>
<summary>✅ Solution</summary>

```python
predictions = model.predict(test[['vol_20d']])
model_rmse = np.sqrt(mean_squared_error(test['vol_next'], predictions))

print('model      :', round(model_rmse, 5))
print('persistence:', part4_pers_rmse)
print('average    :', part4_base_rmse)
```

0.00402 against 0.00417 and 0.00514. The model is best of the three, by about 4% over persistence.

Hold the celebration for now. That number came from the test rows, which is fine because no choice has been made yet. From Q6 onwards choices start, and then it stops being fine.

</details>

---

### Q5 · Clear the bar Part 4 set

Part 4 measured the persistence rule at $R^2 = 0.340$ and called it the bar. Compute the same $R^2$ for your model.

$$R^2 = 1-\frac{\sum_i (y_i-\hat{y}_i)^2}{\sum_i (y_i-\bar{y}_{\text{train}})^2}$$

The denominator uses the **training** mean, because that is what you would have predicted with no model at all.

In [ ]:
rss = ...
tss = ...
model_r2 = ...

print('model R2      :', ...)
print('the bar (Part 4):', part4_pers_r2)

<details>
<summary>💡 Hint 1</summary>

`rss` is `((test['vol_next'] - predictions) ** 2).sum()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`tss` is the same with `train['vol_next'].mean()` in place of `predictions`.

</details>

<details>
<summary>✅ Solution</summary>

```python
rss = ((test['vol_next'] - predictions) ** 2).sum()
tss = ((test['vol_next'] - train['vol_next'].mean()) ** 2).sum()
model_r2 = 1 - rss / tss

print('model R2      :', round(model_r2, 3))
print('the bar (Part 4):', part4_pers_r2)
```

0.388 against a bar of 0.340. Cleared, and not by much.

A model that takes a fitting procedure and a feature and beats a one-line rule by four hundredths of an $R^2$ is a normal result in this subject. Anything dramatically better would be worth checking for a leak.

</details>

---

### Q6 · Three candidates

You used one feature. Part 4 built three. Define the three candidate feature sets, then score each one on the **training** rows.

- `set_a`: `vol_20d` alone
- `set_b`: `vol_20d` and `ret_20d`
- `set_c`: all three

Which fits the training rows best?

In [ ]:
set_a = ...
set_b = ...
set_c = ...

for columns in [set_a, set_b, set_c]:
    ...
    print(columns, ...)

<details>
<summary>💡 Hint 1</summary>

Each set is a plain list of column names, for example `['vol_20d']`.

</details>

<details>
<summary>💡 Hint 2</summary>

Inside the loop: fit on `train[columns]`, predict on `train[columns]`, and score against `train['vol_next']`.

</details>

<details>
<summary>✅ Solution</summary>

```python
set_a = ['vol_20d']
set_b = ['vol_20d', 'ret_20d']
set_c = ['vol_20d', 'ret_20d', 'up_20d']

for columns in [set_a, set_b, set_c]:
    m = LinearRegression()
    m.fit(train[columns], train['vol_next'])
    error = np.sqrt(mean_squared_error(train['vol_next'], m.predict(train[columns])))
    print(columns, round(error, 5))
```

0.00717, 0.00705, 0.00704. More features, lower error, as always. Adding a column can never make a fit worse, so this ordering was guaranteed before you ran anything.

</details>

---

### Q7 · The tempting mistake

Score the same three candidates on the **test** block instead, and see which wins.

Then read the note underneath carefully, because this is the question the whole notebook turns on.

In [ ]:
for columns in [set_a, set_b, set_c]:
    ...
    print(columns, ...)

<details>
<summary>💡 Hint</summary>

The same loop as Q6 with the scoring moved to `test`. The fit stays on `train`.

</details>

<details>
<summary>✅ Solution</summary>

```python
for columns in [set_a, set_b, set_c]:
    m = LinearRegression()
    m.fit(train[columns], train['vol_next'])
    error = np.sqrt(mean_squared_error(test['vol_next'], m.predict(test[columns])))
    print(columns, round(error, 5))
```

0.00402, 0.00424, 0.00424. The ranking reverses: one feature is best and the extra columns were fitting noise.

**And you are now in trouble.** Those three numbers were used to pick a model, so the winner's 0.00402 is no longer an honest estimate of anything. Part of being lowest of three is being genuinely better and part of it is luck, and there is no way to tell how much of each.

Three candidates is mild. A real project compares dozens, and every comparison is another look. The rest of this notebook does the job without them.

</details>

---

### Q8 · A block for choosing

Split the **training** block again by date: fit on everything up to the end of 2020, and compare the candidates on 2021 and 2022. Call them `fit_rows` and `val_rows`.

Which candidate wins now?

In [ ]:
fit_rows = ...
val_rows = ...

for columns in [set_a, set_b, set_c]:
    ...
    print(columns, ...)

<details>
<summary>💡 Hint 1</summary>

`train.loc[:'2020-12-31']` and `train.loc['2021-01-01':]`.

</details>

<details>
<summary>💡 Hint 2</summary>

Fit on `fit_rows`, score on `val_rows`. The test block is not mentioned anywhere in this cell.

</details>

<details>
<summary>✅ Solution</summary>

```python
fit_rows = train.loc[:'2020-12-31']
val_rows = train.loc['2021-01-01':]

for columns in [set_a, set_b, set_c]:
    m = LinearRegression()
    m.fit(fit_rows[columns], fit_rows['vol_next'])
    error = np.sqrt(mean_squared_error(val_rows['vol_next'], m.predict(val_rows[columns])))
    print(columns, round(error, 5))
```

0.00516, 0.00505, 0.00506. Two features win, and the test block played no part in it.

Note that this is not the answer the test rows gave in Q7. Hold that thought.

</details>

---

### Q9 · Move the cut

Part 4 warned that one split is one experiment. Test it. Run the same comparison with the cut at the end of **2019** instead.

In [ ]:
early_fit = ...
early_val = ...

for columns in [set_a, set_b, set_c]:
    ...
    print(columns, ...)

<details>
<summary>💡 Hint</summary>

Only the two dates change: `train.loc[:'2019-12-31']` and `train.loc['2020-01-01':]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
early_fit = train.loc[:'2019-12-31']
early_val = train.loc['2020-01-01':]

for columns in [set_a, set_b, set_c]:
    m = LinearRegression()
    m.fit(early_fit[columns], early_fit['vol_next'])
    error = np.sqrt(mean_squared_error(early_val['vol_next'], m.predict(early_val[columns])))
    print(columns, round(error, 5))
```

0.01003, 0.01070, 0.01095. One feature wins now, clearly.

The candidates did not change. The training block did not change. Only the cut date moved, and the answer moved with it. This is exactly the defect Part 4 listed and could not fix.

</details>

---

### Q10 · Every cut date at once

Instead of choosing a cut date, use five of them and average. Cross-validate the three candidates on the training block with `TimeSeriesSplit`, so every scored block comes after the rows it was fitted on.

$$\text{CV}_K=\frac{1}{K}\sum_{k=1}^{K} e_k$$

Store the three averages in a dictionary called `cv_scores`.

In [ ]:
folds = ...
cv_scores = {}

for name, columns in [('a', set_a), ('b', set_b), ('c', set_c)]:
    ...

cv_scores

<details>
<summary>💡 Hint 1</summary>

`TimeSeriesSplit(n_splits=5)`, then `cross_val_score(LinearRegression(), train[columns], train['vol_next'], cv=folds, scoring='neg_root_mean_squared_error')`.

</details>

<details>
<summary>💡 Hint 2</summary>

The scores come back negative, because scikit-learn reports everything so that larger is better. Store `-scores.mean()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
folds = TimeSeriesSplit(n_splits=5)
cv_scores = {}

for name, columns in [('a', set_a), ('b', set_b), ('c', set_c)]:
    scores = cross_val_score(LinearRegression(), train[columns], train['vol_next'],
                             cv=folds, scoring='neg_root_mean_squared_error')
    cv_scores[name] = round(float(-scores.mean()), 5)

cv_scores
```

0.00736, 0.00742, 0.00750. One feature wins, and the answer is now an average over five cut dates rather than whatever one arbitrary cut happened to say.

Every number is larger than the test RMSE in Q7, and that is expected: each fold fits on fewer rows than the final model will, and scores on an older, harder stretch of history.

</details>

---

### Q11 · Read the folds, not just the average

An average can hide a disaster. Print the five fold errors for `set_a`, and find which period the worst one was scored on.

In [ ]:
scores = ...
fold_errors = ...

worst = ...
blocks = ...

print('folds:', ...)
print('worst fold:', ...)
print('scored from', ..., 'to', ...)

<details>
<summary>💡 Hint 1</summary>

`fold_errors = -scores` is the array of five errors, and `fold_errors.argmax()` gives the position of the largest.

</details>

<details>
<summary>💡 Hint 2</summary>

`blocks = list(folds.split(train))`. Each entry is a pair, and the second half holds the scored row positions, so `train.index[blocks[worst][1][0]]` is its first date.

</details>

<details>
<summary>✅ Solution</summary>

```python
scores = cross_val_score(LinearRegression(), train[set_a], train['vol_next'],
                         cv=folds, scoring='neg_root_mean_squared_error')
fold_errors = -scores

worst = fold_errors.argmax()
blocks = list(folds.split(train))
scored_rows = blocks[worst][1]

print('folds:', fold_errors.round(5))
print('worst fold:', worst + 1)
print('scored from', train.index[scored_rows[0]].date(), 'to', train.index[scored_rows[-1]].date())
```

Fold 3 is roughly twice as bad as the rest, scored from 2019-01-18 to 2020-05-13. It contains March 2020.

A model fitted on the quiet years before it had never seen a month like that. This is worth reporting next to the average, because a risk model that fails precisely when risk arrives is a different object from one that is mediocre throughout.

</details>

---

### Q12 · The folds most people would have used

Run the same comparison with `KFold(n_splits=5, shuffle=True, random_state=0)`, the version of cross-validation found in most textbooks. Store the winners of both methods in a dictionary called `verdicts`.

In [ ]:
shuffled = ...
shuffled_scores = {}

for name, columns in [('a', set_a), ('b', set_b), ('c', set_c)]:
    ...

verdicts = {
    'TimeSeriesSplit': ...,
    'shuffled KFold': ...,
}
verdicts

<details>
<summary>💡 Hint 1</summary>

Only the `cv=` object changes from Q10.

</details>

<details>
<summary>💡 Hint 2</summary>

`min(d, key=d.get)` returns the key with the smallest value.

</details>

<details>
<summary>✅ Solution</summary>

```python
shuffled = KFold(n_splits=5, shuffle=True, random_state=0)
shuffled_scores = {}

for name, columns in [('a', set_a), ('b', set_b), ('c', set_c)]:
    scores = cross_val_score(LinearRegression(), train[columns], train['vol_next'],
                             cv=shuffled, scoring='neg_root_mean_squared_error')
    shuffled_scores[name] = round(float(-scores.mean()), 5)

verdicts = {
    'TimeSeriesSplit': min(cv_scores, key=cv_scores.get),
    'shuffled KFold': min(shuffled_scores, key=shuffled_scores.get),
}
verdicts
```

Shuffled folds give 0.00717, 0.00705, 0.00704, so they choose three features. Time-ordered folds choose one.

Two neighbouring rows share nineteen of their twenty days in both the feature and the target. Shuffling puts one in the fitting block and the other in the scored block, so the model is asked about a row it has effectively already seen. Every score comes back flattering, and the most flexible candidate benefits most.

This is the same dependence Part 4 listed as the first thing still wrong with the setup. It has not gone away; the folds are simply built so that it cannot do any damage.

</details>

---

### Q13 · Close the last gap

One overlap survives even in time-ordered folds. The last rows a fold fits on have targets reaching twenty days into the block about to be scored. Drop them with the `gap` argument, and see what it costs.

In [ ]:
gapped = ...

# then cross-validate set_a with `gapped` and print the average
...

<details>
<summary>💡 Hint</summary>

`TimeSeriesSplit(n_splits=5, gap=20)`, and twenty is the length of the target window.

</details>

<details>
<summary>✅ Solution</summary>

```python
gapped = TimeSeriesSplit(n_splits=5, gap=20)

scores = cross_val_score(LinearRegression(), train[set_a], train['vol_next'],
                         cv=gapped, scoring='neg_root_mean_squared_error')
print('no gap:', cv_scores['a'])
print('gap 20:', round(float(-scores.mean()), 5))
```

0.00736 against 0.00738. A small change, because twenty rows are little against fitting blocks of several hundred.

It moves in the direction it has to: removing a leak can only make an estimate worse, never better. If a correction like this ever improves your score, you have implemented it backwards.

</details>

---

### Q14 · Refit, and open the test block once

The choosing is finished. Cross-validation chose `set_a`. Fit it on **all** the training rows, including the ones the folds used for scoring, and score it once on the test block.

In [ ]:
final_model = ...
final_pred = ...
final_rmse = ...

print('test RMSE  :', ...)
print('persistence:', part4_pers_rmse)
print('average    :', part4_base_rmse)

<details>
<summary>💡 Hint</summary>

Nothing new here. Fit `LinearRegression()` on `train[set_a]`, predict on `test[set_a]`, take the root mean squared error.

</details>

<details>
<summary>✅ Solution</summary>

```python
final_model = LinearRegression()
final_model.fit(train[set_a], train['vol_next'])
final_pred = final_model.predict(test[set_a])
final_rmse = np.sqrt(mean_squared_error(test['vol_next'], final_pred))

print('test RMSE  :', round(final_rmse, 5))
print('persistence:', part4_pers_rmse)
print('average    :', part4_base_rmse)
```

0.00402, the same number Q4 produced, and now it means something different.

In Q4 it was the score of a model nobody had chosen. Here it is the score of a model chosen by a procedure that never saw 2023 or 2024. The arithmetic is identical; the claim you are entitled to make is not.

</details>

---

### Q15 · Does it help when it matters

A risk model exists for the turbulent half of the sample. So far every score has averaged the calm days and the busy ones together.

Split the test block with a **boolean mask**: days whose `vol_20d` is below its median, and the rest. Then score the model and the persistence rule separately on each half.

In [ ]:
model_errors = ...
pers_errors = ...
calm = ...

print('calm, model      :', ...)
print('calm, persistence:', ...)
print('busy, model      :', ...)
print('busy, persistence:', ...)

<details>
<summary>💡 Hint 1</summary>

The two error columns are `test['vol_next'] - final_pred` and `test['vol_next'] - test['vol_20d']`. Subtracting whole columns at once is the vectorised arithmetic from Session 3.

</details>

<details>
<summary>💡 Hint 2</summary>

`calm = test['vol_20d'] < test['vol_20d'].median()` is a mask of True and False, and `~calm` is its opposite. `errors[calm]` keeps one half.

</details>

<details>
<summary>✅ Solution</summary>

```python
model_errors = test['vol_next'] - final_pred
pers_errors = test['vol_next'] - test['vol_20d']
calm = test['vol_20d'] < test['vol_20d'].median()

print('calm, model      :', round(np.sqrt((model_errors[calm] ** 2).mean()), 5))
print('calm, persistence:', round(np.sqrt((pers_errors[calm] ** 2).mean()), 5))
print('busy, model      :', round(np.sqrt((model_errors[~calm] ** 2).mean()), 5))
print('busy, persistence:', round(np.sqrt((pers_errors[~calm] ** 2).mean()), 5))
```

On the 241 calmest test days the model beats persistence by 8%, 0.00375 against 0.00408. On the busy half the two are level: 0.00426 against 0.00426.

**The whole of the model's advantage comes from the quiet days.** Those are the days a risk report needs a forecast for least.

This is not a reason to throw the model away, and it does not undo Q14. It is the first line of the limitations section, and you found it with one mask and no loop. A single RMSE averages the two halves together and shows none of it.

</details>

---

### Q16 · The test Part 4's rule failed

Part 4 took the persistence rule across all eleven instruments and found it positive on only 4 of them. Do the same with the model.

For every ticker, build the two-column table, split it at the end of 2022, fit `vol_20d` on the training rows, and record the model's test RMSE next to the persistence rule's. Store the pairs in a dictionary called `across_desk`.

In [ ]:
across_desk = {}

for ticker in TICKERS:
    ...

beat = ...
print('model beats persistence on', ..., 'of', ...)

<details>
<summary>💡 Hint 1</summary>

Inside the loop, build `frame` with `vol_20d` and `vol_next` only, then `.dropna()` and split it the same way as Q2.

</details>

<details>
<summary>💡 Hint 2</summary>

Store a pair: `across_desk[ticker] = (model_rmse, pers_rmse)`. Then `sum(1 for t in across_desk if across_desk[t][0] < across_desk[t][1])`.

</details>

<details>
<summary>✅ Solution</summary>

```python
across_desk = {}

for ticker in TICKERS:
    frame = pd.DataFrame({'vol_20d': rets[ticker].rolling(20).std()})
    frame['vol_next'] = rets[ticker].rolling(20).std().shift(-20)
    frame = frame.dropna()

    tr = frame.loc[:'2022-12-31']
    te = frame.loc['2023-01-01':]

    m = LinearRegression()
    m.fit(tr[['vol_20d']], tr['vol_next'])
    model_rmse = np.sqrt(mean_squared_error(te['vol_next'], m.predict(te[['vol_20d']])))
    pers_rmse = np.sqrt(mean_squared_error(te['vol_next'], te['vol_20d']))
    across_desk[ticker] = (model_rmse, pers_rmse)

beat = sum(1 for t in across_desk if across_desk[t][0] < across_desk[t][1])
print('model beats persistence on', beat, 'of', len(across_desk))
```

**11 of 11.**

This is the most important number in the notebook. Part 4 ended with a rule that looked convincing on Apple and was positive on 4 of eleven names. A model chosen by cross-validation on one stock, with the choice made without ever looking at 2023 or 2024, transfers to the whole desk.

That is what separates a result from a coincidence, and it took five sessions of setup to be able to say it honestly.

</details>

---

### Q17 · Draw it

Draw the improvement, `persistence - model`, for the eleven instruments, sorted from largest to smallest, with a line at zero.

In [ ]:
improvement = ...
ranked = ...

fig, ax = plt.subplots(figsize=(9, 3.2))
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`across_desk[t]` is a pair, so the improvement is `across_desk[t][1] - across_desk[t][0]`.

</details>

<details>
<summary>💡 Hint 2</summary>

`pd.Series(improvement).sort_values(ascending=False)`, then `ax.bar(...)` and `ax.axhline(0, color='black')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
improvement = {t: across_desk[t][1] - across_desk[t][0] for t in across_desk}
ranked = pd.Series(improvement).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(ranked.index, ranked.values)
ax.axhline(0, color='black', linewidth=1)
ax.set_ylabel('RMSE saved')
ax.set_title('How much the model beats persistence by, 2023 to 2024', loc='left')
plt.show()
```

Every bar sits above the line. JNJ gains the most in relative terms, about 25% off the persistence error.

Compare this with the equivalent figure in Part 3, where the ranking of volatility barely carried from one period to the next. A forecast that helps everywhere is a very different object from a description that held once.

</details>

---

### Q18 · Write down what you would defend

Finish the investigation the way Part 4 finished: one dictionary that somebody else could pick up, and a function that prints it with a verdict.

Fill in `report`, then write `summarise(report)`: it prints each entry on its own line and ends with one sentence on whether the model earned its place.

A dictionary, a loop, an f-string and an `if`. All of it comes from Sessions 1 and 2.

In [ ]:
report = {
    'target': ...,
    'features_kept': ...,
    'features_tried': ...,
    'chosen_by': ...,
    'folds': ...,
    'test_rmse': ...,
    'persistence_rmse': ...,
    'beat_across_desk': ...,
}

def summarise(report):
    ...

summarise(report)

<details>
<summary>💡 Hint 1</summary>

Most of the values are already in variables: `final_rmse` from Q14, `part4_pers_rmse` from the quick load, and `beat` from Q16.

</details>

<details>
<summary>💡 Hint 2</summary>

Inside the function, `for key in report:` then `print(f'{key:18} {report[key]}')`. For the verdict, an `if` on whether `test_rmse` is below `persistence_rmse`.

</details>

<details>
<summary>✅ Solution</summary>

```python
report = {
    'target': part4_target,
    'features_kept': set_a,
    'features_tried': part4_features,
    'chosen_by': 'cross-validation on the training block only',
    'folds': 'TimeSeriesSplit, 5 splits, expanding window',
    'test_rmse': round(final_rmse, 5),
    'persistence_rmse': part4_pers_rmse,
    'beat_across_desk': f'{beat} of {len(across_desk)}',
}

def summarise(report):
    """Print a finished model selection, and say whether it earned its place."""
    for key in report:
        print(f'{key:18} {report[key]}')

    if report['test_rmse'] < report['persistence_rmse']:
        print('\nThe model beats the no-model rule on data used in no decision.')
    else:
        print('\nThe model does not beat the no-model rule. Report the rule.')

summarise(report)
```

Eight lines that another analyst could act on. Note what is in there besides the score: which features were **tried** as well as which were kept, and how the choice was made.

A reader who knows only that `vol_20d` was kept cannot tell whether it won against two rivals or against two hundred. That difference is the whole content of this session, and it belongs in the report.

</details>

---

## 🧭 What you have now

| what | where it lives |
|:--|:--|
| the learning table, rebuilt | `table`, `train`, `test` |
| the first fitted model | `model`, `final_model` |
| its honest test score | `final_rmse` |
| the three candidates | `set_a`, `set_b`, `set_c` |
| what one validation block said | Q8 and Q9, and they disagree |
| what five time-ordered folds said | `cv_scores` |
| what shuffled folds would have said | `verdicts` |
| where the advantage comes from | `calm`, and the two halves of Q15 |
| the whole desk | `across_desk` |
| the thing you would defend | `report` |

## What changed since Part 4

Part 4 ended with three complaints about its own setup. Two of them now have answers.

- **One split was one experiment.** Q9 showed the winner changing when the cut moved, and Q10 replaced the single cut with five and an average.
- **The rows were not independent.** Q12 showed what that dependence does to shuffled folds, and Q13 closed the last overlap at the fold boundary. The dependence is still there; the folds are simply built so that it cannot flatter the score.
- **The regimes differ.** This one is not solved and probably cannot be. Q11 found it sitting in fold 3, where the error is twice the others because that block contains March 2020, and Q15 found it again in the test years, where the model's advantage over persistence is entirely in the calm half. The honest response is to report those numbers next to the average rather than to hide behind it.

## Where this leaves the risk report

Five parts ago this was two stocks compared with a subtraction. It is now a forecast, with a model chosen by a procedure that never saw the years it is reported on, beating a no-model rule on 11 of 11 instruments.

The model itself is the least interesting thing in it. One feature and two coefficients, and the slope near a half that says volatility reverts. Everything that makes the number believable is the apparatus around it.

**Next block:** models that bring settings of their own, and a search over those settings that uses exactly the folds you built here.